# SalesInsight — Pipeline de Análise de Vendas

# ARQUIVO GERADO EM JUPYTER CONFORME CODIGO DO MINIPROJETO
**Módulo 1, Semana 08 — SCTEC**

| RF | Função / Classe | Descrição |
|---|---|---|
| RF01 | `gerar_dataset_vendas` | Dataset sintético com dados sujos |
| RF02 | `inspecionar_dados` | Inspeção e diagnóstico |
| RF03 | `limpar_dados` | Limpeza e normalização de tipos |
| RF04 | `criar_colunas_derivadas` | Feature engineering com `np.select` |
| RF05 | `calcular_metricas` | Agregações com `groupby` |
| RF06 | `segmentar_clientes` | Classificação Bronze/Prata/Ouro |
| RF07 | `calcular_estatisticas_numpy` | Estatísticas vetorizadas |
| RF08 | `gerar_visualizacoes` | Gráficos inline (matplotlib/seaborn) |
| RF09 | `AnalisadorDeVendas` | Classe com method chaining |
| RF10 | `AnalisadorComProjecao` | Herança + `super()` + projeções |
| RF11 | `processar_coluna` | Higher-order function + lambda |
| RF12 | `exportar_resultados` | Exportação CSV e JSON |
| RF13 | `limpar_strings_com_regex` | Limpeza com `re.sub` e `re.compile` |
| RF14 | `main` | Orquestra o pipeline completo |

## Imports e Configuração

In [1]:
import os
import re
import json
import random
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

%matplotlib inline

## Constantes do Domínio

## RF01 — Geração do Dataset Sintético

In [2]:
def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintético de vendas com 200 registros, dados intencionalmente sujos, seed=42 fixa a sequencia
       de numeros aleatorios"""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor", "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares", "Tablet": "Celulares",
                  "Monitor": "Computadores", "Teclado": "Periféricos", "Mouse": "Periféricos",
                  "Headset": "Periféricos"}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    
    """ Gera uma string formatada onde i = numero atual do laço, e "":03d"  significa 
        d --> numero inteiro decimal
        3 --> ocupar 3 posições
        0 --> completar com zeros à esquerda

        i = 25 --> "025"
    """
    clientes = [f"Cliente_{i:03d}" for i in range(1, 51)]

    data_inicio = datetime.datetime(2024, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        quantidade = random.randint(1, 10)
        preco_base = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
                      "Monitor": 1200, "Teclado": 250, "Mouse": 120, "Headset": 350}[produto]
        preco = round(preco_base * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))

    # Inserindo dados intencionalmente sujos para limpeza
        if random.random() < 0.05:
            quantidade = None          # valor nulo
        if random.random() < 0.04:
            preco = None               # valor nulo
        if random.random() < 0.03:
            produto = "  " + produto   # espaço extra (string suja)
        dados.append({
            "id_venda": i + 1,
            "data_venda": data.strftime("%Y-%m-%d") if random.random() > 0.02 else "DATA INVÁLIDA",
            "cliente": random.choice(clientes),
            "produto": produto,
            "categoria": categorias.get(produto.strip(), "Outros"),
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco
        })

    return pd.DataFrame(dados)

    # Gerar e salvar
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendasjupiter.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())


Dataset gerado com 200 registros.
   id_venda  data_venda      cliente     produto     categoria    regiao  \
0         1  2024-05-20  Cliente_035       Mouse   Periféricos   Sudeste   
1         2  2024-02-17  Cliente_042     Teclado   Periféricos     Norte   
2         3  2024-05-22  Cliente_022     Monitor  Computadores  Nordeste   
3         4  2024-06-21  Cliente_017  Smartphone     Celulares   Sudeste   
4         5  2024-07-12  Cliente_024       Mouse   Periféricos     Norte   

   quantidade  preco_unitario  
0         2.0          102.90  
1         7.0          214.88  
2         4.0             NaN  
3         4.0         2501.76  
4         8.0          121.30  


## RF02 — Inspeção dos Dados

In [3]:
def inspecionar_dados(df):
    """Exibe informações básicas do DataFrame."""
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos de dados:\n{df.dtypes}")
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df.head()}")
    print(f"\nEstatísticas descritivas:\n{df.describe()}")

inspecionar_dados(df_bruto)



=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade         6
preco_unitario    10
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente     produto     categoria    regiao  \
0         1  2024-05-20  Cliente_035       Mouse   Periféricos   Sudeste   
1         2  2024-02-17  Cliente_042     Teclado   Periféricos     Norte   
2         3  2024-05-22  Cliente_022     Monitor  Computadores  Nordeste   
3         4  2024-06-21  Cliente_017  Smartphone     Celular

## RF03 — Limpeza e Tratamento de Dados

In [4]:
import re

def limpar_dados(df):
    """
    Limpa e trata o DataFrame de vendas.
    Retorna o DataFrame limpo e um relatório de limpeza.
    """
    n_inicial = len(df)
    relatorio = {}

    # 1. Remover espaços extras em colunas de texto
    colunas_texto = df.select_dtypes(include=["string", "object"]).columns
    for col in colunas_texto:
        df[col] = df[col].str.strip()

    # 2. Converter data e remover datas inválidas
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    n_datas_invalidas = df["data_venda"].isnull().sum()
    df = df.dropna(subset=["data_venda"])
    relatorio["datas_invalidas_removidas"] = n_datas_invalidas

    # 3. Remover linhas com quantidade ou preço nulos
    n_antes = len(df)
    df = df.dropna(subset=["quantidade", "preco_unitario"])
    relatorio["linhas_nulas_removidas"] = n_antes - len(df)

    # 4. Garantir tipos numéricos corretos
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    n_final = len(df)
    relatorio["registros_iniciais"] = n_inicial
    relatorio["registros_finais"] = n_final
    relatorio["registros_removidos_total"] = n_inicial - n_final

    print("\n=== RELATÓRIO DE LIMPEZA ===")
    for chave, valor in relatorio.items():
        print(f"  {chave}: {valor}")

limpar_dados(df_bruto)

df_limpo = df_bruto


=== RELATÓRIO DE LIMPEZA ===
  datas_invalidas_removidas: 4
  linhas_nulas_removidas: 16
  registros_iniciais: 200
  registros_finais: 180
  registros_removidos_total: 20


## RF04 — Colunas Derivadas com `np.select`

In [5]:
def criar_colunas_derivadas(df):
    """Cria colunas calculadas e derivadas a partir do dataset limpo."""

    # Receita total por linha de venda
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]

    # Extração de componentes de data
    df["mes"] = df["data_venda"].dt.month
    
    meses = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março",
    4: "Abril", 5: "Maio", 6: "Junho",
    7: "Julho", 8: "Agosto", 9: "Setembro",
    10: "Outubro", 11: "Novembro", 12: "Dezembro"
}   
    df["mes_nome"] = df["data_venda"].dt.month.map(meses) # nome do mês em portugues
    
    df["trimestre"] = df["data_venda"].dt.quarter.apply(lambda q: f"Q{q}")
    df["ano"] = df["data_venda"].dt.year

    # Classificação da receita por item com numpy.select (transformação condicional vetorizada)
    condicoes = [
    df["receita_total"] < 500,
    (df["receita_total"] >= 500) & (df["receita_total"] < 5000),
    df["receita_total"] >= 5000
    ]
    classificacoes = ["Baixo Valor", "Médio Valor", "Alto Valor"]
    df["faixa_receita_item"] = np.select(condicoes, classificacoes, default="Não Classificado")
    print("\n=== COLUNAS DERIVADAS CRIADAS ===")
    print(df[["data_venda", "receita_total", "mes", "trimestre", "faixa_receita_item"]].head())

    return df

criar_colunas_derivadas(df_limpo)

df_derivadas = df_limpo


=== COLUNAS DERIVADAS CRIADAS ===
  data_venda  receita_total  mes trimestre faixa_receita_item
0 2024-05-20         205.80  5.0      Q2.0        Baixo Valor
1 2024-02-17        1504.16  2.0      Q1.0        Médio Valor
2 2024-05-22            NaN  5.0      Q2.0   Não Classificado
3 2024-06-21       10007.04  6.0      Q2.0         Alto Valor
4 2024-07-12         970.40  7.0      Q3.0        Médio Valor


## RF05 — Métricas Agregadas com `groupby`

In [6]:
def calcular_metricas(df):
    """Calcula e retorna métricas agregadas do dataset."""
    metricas = {}

    # Receita por mês
    por_mes = df.groupby(["mes","mes_nome"]).agg(
        receita_total=("receita_total", "sum"),
        quantidade=("quantidade", "sum"),
        n_vendas=("id_venda", "count")
    ).reset_index().sort_values("mes")
    metricas["por_mes"] = por_mes

    # Top 5 produtos por receita
    top_produtos = df.groupby("produto")["receita_total"].sum()\
                     .sort_values(ascending=False).head(5).reset_index()
    metricas["top_produtos"] = top_produtos

    # Receita por categoria
    por_categoria = df.groupby("categoria")["receita_total"].sum().reset_index()
    metricas["por_categoria"] = por_categoria

    # Receita por região
    por_regiao = df.groupby("regiao").agg(
        receita_total=("receita_total", "sum"),
        media_ticket=("receita_total", "mean")
    ).reset_index().sort_values("receita_total", ascending=False)
    por_regiao["media_ticket"] = por_regiao["media_ticket"].round(2)
    metricas["por_regiao"] = por_regiao

    # Exibição
    for nome, tabela in metricas.items():
        print(f"\n=== {nome.upper().replace('_', ' ')} ===")
        print(tabela.to_string(index=False))

    return metricas

calcular_metricas(df_derivadas)

metricas = calcular_metricas(df_derivadas)


=== POR MES ===
 mes  mes_nome  receita_total  quantidade  n_vendas
 1.0   Janeiro       47169.34        45.0        10
 2.0 Fevereiro       69968.85        94.0        16
 3.0     Março      124532.20        91.0        19
 4.0     Abril      103611.52        75.0        15
 5.0      Maio      106602.03       100.0        17
 6.0     Junho      117334.85        75.0        16
 7.0     Julho      110935.66        99.0        19
 8.0    Agosto      140320.55        81.0        15
 9.0  Setembro       90627.26        66.0        11
10.0   Outubro      137350.13       127.0        21
11.0  Novembro       69554.97        65.0        13
12.0  Dezembro      115925.56       122.0        24

=== TOP PRODUTOS ===
   produto  receita_total
  Notebook      354209.68
    Tablet      353176.81
Smartphone      234977.18
   Monitor      191560.87
   Headset       59314.70

=== POR CATEGORIA ===
   categoria  receita_total
   Celulares      588153.99
Computadores      545770.55
 Periféricos      1076

## RF06 — Segmentação de Clientes (Bronze / Prata / Ouro)

In [7]:
def segmentar_clientes(df):
    """Segmenta clientes pelo total gasto usando groupby e lambda."""

    clientes = df.groupby("cliente")["receita_total"].sum().reset_index()
    clientes.columns = ["cliente", "total_gasto"]

    # Classificação usando função lambda com condicionais
    clientes["segmento"] = clientes["total_gasto"].apply(
        lambda gasto: "Ouro" if gasto > 15000
                      else ("Prata" if gasto >= 5000 else "Bronze")
    )

    clientes = clientes.sort_values("total_gasto", ascending=False)

    print("\n=== SEGMENTAÇÃO DE CLIENTES ===")
    print(clientes.head(10).to_string(index=False))
    print(f"\nDistribuição de segmentos:\n{clientes['segmento'].value_counts()}")

    return clientes

segmentar_clientes(df_derivadas)


=== SEGMENTAÇÃO DE CLIENTES ===
    cliente  total_gasto segmento
Cliente_015     82964.76     Ouro
Cliente_035     75402.52     Ouro
Cliente_048     62293.77     Ouro
Cliente_024     57381.21     Ouro
Cliente_006     50249.74     Ouro
Cliente_004     50106.56     Ouro
Cliente_032     45532.00     Ouro
Cliente_044     43513.88     Ouro
Cliente_016     42754.07     Ouro
Cliente_017     41983.05     Ouro

Distribuição de segmentos:
segmento
Ouro      31
Prata     10
Bronze     8
Name: count, dtype: int64


,cliente,total_gasto,segmento
13,Cliente_015,82964.76,Ouro
33,Cliente_035,75402.52,Ouro
46,Cliente_048,62293.77,Ouro
22,Cliente_024,57381.21,Ouro
5,Cliente_006,50249.74,Ouro
3,Cliente_004,50106.56,Ouro
30,Cliente_032,45532.00,Ouro
42,Cliente_044,43513.88,Ouro
14,Cliente_016,42754.07,Ouro
15,Cliente_017,41983.05,Ouro


## RF07 — Estatísticas com NumPy Vetorizado

In [8]:
def calcular_estatisticas_numpy(df):
    """Usa NumPy para calcular estatísticas sobre as receitas."""
    print("\n=== ESTATÍSTICAS COM NUMPY ===")

    receitas = df["receita_total"].dropna().to_numpy()  # Converte para array NumPy, e dropna retira resultado nan

    media = np.mean(receitas)
    mediana = np.median(receitas)
    desvio_padrao = np.std(receitas)
    total = np.sum(receitas)
    p25 = np.percentile(receitas, 25)
    p75 = np.percentile(receitas, 75)

    print(f"  Receita média por venda:    R$ {media:.2f}")
    print(f"  Receita mediana por venda:  R$ {mediana:.2f}")
    print(f"  Desvio padrão:              R$ {desvio_padrao:.2f}")
    print(f"  Receita total:              R$ {total:.2f}")
    print(f"  Percentil 25 (Q1):          R$ {p25:.2f}")
    print(f"  Percentil 75 (Q3):          R$ {p75:.2f}")

  # Broadcasting: normalizar receitas entre 0 e 1
    receitas_normalizadas = (receitas - receitas.min()) / (receitas.max() - receitas.min())
    print(f"\n  Receitas normalizadas (primeiros 5): {receitas_normalizadas[:5].round(4)}")

    # Operação vetorizada: identificar vendas acima da média sem loop
    acima_da_media = receitas[receitas > media]
    print(f"\n  Vendas acima da média: {len(acima_da_media)} de {len(receitas)}")

    return {
        "media": media, "mediana": mediana,
        "desvio_padrao": desvio_padrao, "total": total
    }

calcular_estatisticas_numpy(df_derivadas)



=== ESTATÍSTICAS COM NUMPY ===
  Receita média por venda:    R$ 6747.83
  Receita mediana por venda:  R$ 3841.00
  Desvio padrão:              R$ 6642.87
  Receita total:              R$ 1241600.77
  Percentil 25 (Q1):          R$ 1215.35
  Percentil 75 (Q3):          R$ 10878.61

  Receitas normalizadas (primeiros 5): [0.     0.042  0.3168 0.0247 0.1162]

  Vendas acima da média: 74 de 184


{'media': np.float64(6747.830271739131),
 'mediana': np.float64(3841.0),
 'desvio_padrao': np.float64(6642.869611190948),
 'total': np.float64(1241600.77)}

## RF08 — Visualizações (matplotlib / seaborn)



In [9]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_visualizacoes(df, metricas, output_dir="outputs/graficos/jupiter"):
    """Gera e exporta visualizações dos dados de vendas."""
    os.makedirs(output_dir, exist_ok=True)

    # Configurações visuais globais
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 6)
    plt.rcParams["axes.titlesize"] = 14
    plt.rcParams["axes.labelsize"] = 12

    # --- Gráfico 1: Receita por Mês (linha) ---
    fig, ax = plt.subplots()
    por_mes = metricas["por_mes"]
    ax.plot(por_mes["mes"], por_mes["receita_total"], marker="o", linewidth=2, color="#2196F3")
    ax.fill_between(por_mes["mes"], por_mes["receita_total"], alpha=0.15, color="#2196F3")
    ax.set_title("Receita Total por Mês (2024)")
    ax.set_xlabel("Mês")
    ax.set_ylabel("Receita Total (R$)")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun",
                         "Jul","Ago","Set","Out","Nov","Dez"], rotation=45)
    plt.tight_layout()
    caminho = os.path.join(output_dir, "vendas_por_mes.png")
    plt.savefig(caminho, dpi=150)
    plt.close()
    print(f"  Gráfico exportado: {caminho}")

    # --- Gráfico 2: Top 5 Produtos (barras horizontais) ---
    fig, ax = plt.subplots()
    top = metricas["top_produtos"]
    sns.barplot(data=top, y="produto", x="receita_total", ax=ax, palette="Blues_d")
    ax.set_title("Top 5 Produtos por Receita Total")
    ax.set_xlabel("Receita Total (R$)")
    ax.set_ylabel("Produto")
    for container in ax.containers:
        ax.bar_label(container, fmt="R$ %.0f", padding=5)
    plt.tight_layout()
    caminho = os.path.join(output_dir, "top_produtos.png")
    plt.savefig(caminho, dpi=150)
    plt.close()
    print(f"  Gráfico exportado: {caminho}")

    # --- Gráfico 3: Distribuição de Receita por Região (boxplot) ---
    fig, ax = plt.subplots()
    sns.boxplot(data=df, x="regiao", y="receita_total", ax=ax, palette="Set2")
    ax.set_title("Distribuição de Receita por Transação – Por Região")
    ax.set_xlabel("Região")
    ax.set_ylabel("Receita por Venda (R$)")
    plt.xticks(rotation=30)
    plt.tight_layout()
    caminho = os.path.join(output_dir, "distribuicao_regioes.png")
    plt.savefig(caminho, dpi=150)
    plt.close()
    print(f"  Gráfico exportado: {caminho}")

    print("\n=== VISUALIZAÇÕES GERADAS COM SUCESSO ===")

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

gerar_visualizacoes(df_derivadas, metricas)



  Gráfico exportado: outputs/graficos/jupiter\vendas_por_mes.png
  Gráfico exportado: outputs/graficos/jupiter\top_produtos.png
  Gráfico exportado: outputs/graficos/jupiter\distribuicao_regioes.png

=== VISUALIZAÇÕES GERADAS COM SUCESSO ===


## RF09 — Classe `AnalisadorDeVendas` (Method Chaining)

In [10]:
class AnalisadorDeVendas:
    def __init__(self, dados):
        self.dados = dados

    def resumo(self):
        print("Resumo das vendas")
    
    def __init__(self, dados):
        self.dados = dados
        self.metricas = {}
    def analisar(self):

        self.metricas["por_mes"] = (
            self.dados
            .groupby("mes")
            .agg(receita_total=("receita_total", "sum"))
            .reset_index()
        )

        return self
 


class AnalisadorComProjecao(AnalisadorDeVendas):
    """
    Extensão do AnalisadorDeVendas com funcionalidades de projeção simples.
    Herda todos os métodos da classe pai e adiciona projeção de tendência.
    """
    def analisar(self):
        self.metricas = {}

        self.metricas["por_mes"] = (
        self.dados
        .groupby("mes")
        .agg(receita_total=("receita_total", "sum"))
        .reset_index()
    )

        return self

    def __init__(self, caminho_arquivo, meses_projecao=3):
        super().__init__(caminho_arquivo)
        self.meses_projecao = meses_projecao
        self.projecoes = []

    def projetar_tendencia(self):
        """
        Projeta a receita dos próximos meses com base na média móvel dos últimos 3 meses.
        Método simples sem machine learning – baseado em médias.
        """
        if not self.metricas or "por_mes" not in self.metricas:
            print("[AVISO] Rode .analisar() antes de projetar.")
            return self

        por_mes = self.metricas["por_mes"].sort_values("mes")
        receitas_historicas = por_mes["receita_total"].to_numpy()

        # Média móvel dos últimos 3 meses como base da projeção
        ultimos_3 = receitas_historicas[-3:]
        media_movel = np.mean(ultimos_3)
        tendencia = np.std(ultimos_3) * 0.1  # fator de crescimento simples

        ultimo_mes = int(por_mes["mes"].max())

        print("\n=== PROJEÇÃO DE TENDÊNCIA (Média Móvel Simples) ===")
        print(f"  Base: média dos últimos 3 meses = R$ {media_movel:,.2f}")
        self.projecoes = []

        for i in range(1, self.meses_projecao + 1):
            mes_projetado = (ultimo_mes + i - 1) % 12 + 1
            receita_projetada = media_movel + (tendencia * i)
            self.projecoes.append({"mes": mes_projetado, "receita_projetada": round(receita_projetada, 2)})
            print(f"  Mês {mes_projetado:02d} (projeção): R$ {receita_projetada:,.2f}")

        return self

    def exibir_projecao_detalhada(self):
        """Exibe as projeções calculadas."""
        if not self.projecoes:
            print("[AVISO] Nenhuma projeção disponível. Rode .projetar_tendencia() primeiro.")
            return
        print("\n=== DETALHAMENTO DAS PROJEÇÕES ===")
        for p in self.projecoes:
            print(f"  Mês {p['mes']:02d}: R$ {p['receita_projetada']:,.2f}")


analise = AnalisadorComProjecao(df_derivadas)

analise.analisar()

analise.projetar_tendencia()


=== PROJEÇÃO DE TENDÊNCIA (Média Móvel Simples) ===
  Base: média dos últimos 3 meses = R$ 107,610.22
  Mês 01 (projeção): R$ 110,439.71
  Mês 02 (projeção): R$ 113,269.21
  Mês 03 (projeção): R$ 116,098.70


## RF10 — Herança com `AnalisadorComProjecao`

In [11]:

class AnalisadorComProjecao(AnalisadorDeVendas):
    """
    Extensão do AnalisadorDeVendas com funcionalidades de projeção simples.
    Herda todos os métodos da classe pai e adiciona projeção de tendência.
    """

    def __init__(self, caminho_arquivo: str, meses_projecao: int=3):
        super().__init__(caminho_arquivo)
        self.meses_projecao = meses_projecao
        self.projecoes = []

    def projetar_tendencia(self):
        """
        Projeta a receita dos próximos meses com base na média móvel dos últimos 3 meses.
        Método simples sem machine learning – baseado em médias.
        """
        if not self.metricas or "por_mes" not in self.metricas:
            print("[AVISO] Rode .analisar() antes de projetar.")
            return self

        por_mes = self.metricas["por_mes"].sort_values("mes")
        receitas_historicas = por_mes["receita_total"].to_numpy()

        # Média móvel dos últimos 3 meses como base da projeção
        ultimos_3 = receitas_historicas[-3:]
        media_movel = np.mean(ultimos_3)
        tendencia = np.std(ultimos_3) * 0.1  # fator de crescimento simples

        ultimo_mes = int(por_mes["mes"].max())

        print("\n=== PROJEÇÃO DE TENDÊNCIA (Média Móvel Simples) ===")

        print(f"  Base: média dos últimos 3 meses = R$ {media_movel:,.2f}")
       
        self.projecoes = []

        for i in range(1, self.meses_projecao + 1):
            mes_projetado = (ultimo_mes + i - 1) % 12 + 1
            receita_projetada = media_movel + (tendencia * i)
            self.projecoes.append({"mes": mes_projetado, "receita_projetada": round(receita_projetada, 2)})
            print(f"  Mês {mes_projetado:02d} (projeção): R$ {receita_projetada:,.2f}")

        return self

    def exibir_projecao_detalhada(self):
        """Exibe as projeções calculadas."""
        if not self.projecoes:
            print("[AVISO] Nenhuma projeção disponível. Rode .projetar_tendencia() primeiro.")
            return
        print("\n=== DETALHAMENTO DAS PROJEÇÕES ===")
        for p in self.projecoes:
            print(f"  Mês {p['mes']:02d}: R$ {p['receita_projetada']:,.2f}")

analise = AnalisadorComProjecao(df_derivadas)

analise.analisar()

analise.projetar_tendencia()



=== PROJEÇÃO DE TENDÊNCIA (Média Móvel Simples) ===
  Base: média dos últimos 3 meses = R$ 107,610.22
  Mês 01 (projeção): R$ 110,439.71
  Mês 02 (projeção): R$ 113,269.21
  Mês 03 (projeção): R$ 116,098.70


## RF11 — Higher-Order Function + Lambda

In [14]:
# Lambda em apply (transformação condicional de coluna)
df_derivadas["desconto"] = df_derivadas["receita_total"].apply(lambda x: 0.10 if x > 10000 else 0.05)

# Lambda em sorted para ordenar lista de dicionários
lista_produtos = df_derivadas.to_dict("records")

produtos_ordenados = sorted(lista_produtos,key=lambda p: p["receita_total"],  reverse=True)

# Lambda como filtro rápido
vendas_alto_valor = df_derivadas[df_derivadas["receita_total"].apply(lambda x: x > 5000)]

print(df_derivadas['desconto'])

print(produtos_ordenados)

print(vendas_alto_valor)


0      0.05
1      0.05
2      0.05
3      0.10
4      0.05
       ... 
195    0.05
196    0.05
197    0.10
198    0.10
199    0.05
Name: desconto, Length: 200, dtype: float64
[{'id_venda': 3, 'data_venda': Timestamp('2024-05-22 00:00:00'), 'cliente': 'Cliente_022', 'produto': 'Monitor', 'categoria': 'Computadores', 'regiao': 'Nordeste', 'quantidade': 4.0, 'preco_unitario': nan, 'receita_total': nan, 'mes': 5.0, 'mes_nome': 'Maio', 'trimestre': 'Q2.0', 'ano': 2024.0, 'faixa_receita_item': 'Não Classificado', 'desconto': 0.05}, {'id_venda': 17, 'data_venda': Timestamp('2024-09-29 00:00:00'), 'cliente': 'Cliente_017', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': 8.0, 'preco_unitario': 3831.82, 'receita_total': 30654.56, 'mes': 9.0, 'mes_nome': 'Setembro', 'trimestre': 'Q3.0', 'ano': 2024.0, 'faixa_receita_item': 'Alto Valor', 'desconto': 0.1}, {'id_venda': 21, 'data_venda': Timestamp('2024-05-04 00:00:00'), 'cliente': 'Cliente_028', 'produto': 'Sma

## RF12 — Exportação de Resultados (CSV e JSON)

In [ ]:
import json

def exportar_resultados(metricas, clientes, stats_numpy):
    """Exporta resultados em CSV e JSON."""
    os.makedirs("outputs", exist_ok=True)

    # Exportar CSV com métricas por mês
    caminho_csv = "outputs/metricas_por_mes.csv"

    metricas["por_mes"].to_csv(caminho_csv, index=False, encoding="utf-8-sig")
    print(f"  CSV exportado: {caminho_csv}")

    # Exportar segmentação de clientes em CSV
    caminho_clientes = "outputs/segmentacao_clientes.csv"
    clientes.to_csv(caminho_clientes, index=False, encoding="utf-8-sig")
    print(f"  CSV exportado: {caminho_clientes}")

    # Exportar estatísticas gerais em JSON
    caminho_json = "outputs/estatisticas_gerais.json"
    stats_serializaveis = {k: round(float(v), 2) for k, v in stats_numpy.items()}
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(stats_serializaveis, f, indent=4, ensure_ascii=False)
    print(f"  JSON exportado: {caminho_json}")

    # Ler e exibir o JSON exportado para confirmar
    with open(caminho_json, "r", encoding="utf-8") as f:
        dados_lidos = json.load(f)
    print(f"\n  Conteúdo do JSON exportado:\n  {json.dumps(dados_lidos, indent=2)}")




## RF13 — Limpeza de Strings com Regex

In [ ]:
import re

def limpar_strings_com_regex(df):
    """
    Usa expressões regulares para limpeza de colunas de texto.
    Exemplos: remover caracteres especiais, padronizar formatos.
    """
 # 1. Remover caracteres não alfanuméricos do nome do cliente (exceto underline e espaço)
    df["cliente_limpo"] = df["cliente"].apply(
        lambda s: re.sub(r"[^a-zA-Z0-9_ ]", "", str(s)).strip()
    )

    # 2. Identificar registros com padrão de ID inválido (deve ser "Cliente_XXX")
    padrao_cliente = re.compile(r"^Cliente_\d{3}$")
    df["cliente_valido"] = df["cliente_limpo"].apply(
        lambda s: bool(padrao_cliente.match(s))
    )

    n_invalidos = (~df["cliente_valido"]).sum()
    print(f"\n=== LIMPEZA COM REGEX ===")
    print(f"  Clientes com formato inválido encontrados: {n_invalidos}")
    print(f"  Amostra de clientes limpos: {df['cliente_limpo'].head(5).tolist()}")

    return df


## RF14 — Pipeline Principal (`main`)

In [ ]:
def main() -> None:
    """RF14 — Executa o pipeline completo de análise de vendas."""
    print("\n" + "=" * 55)
    print("  SALESINSIGHT IPYNB — Pipeline de Análise de Vendas")
    print("=" * 55)

    # RF01 — Geração
    df_bruto = gerar_dataset_vendas(n_registros=200, seed=42)
    df_bruto.to_csv("vendasjupiter.csv", index=False, encoding="utf-8")
    print(f"  [RF01] Dataset gerado: {df_bruto.shape[0]} registros | salvo em vendas.csv")

    # RF02 — Inspeção
    inspecionar_dados(df_bruto)

    # RF03 — Limpeza
    df_limpo, relatorio = limpar_dados(df_bruto)

    # RF04 — Feature engineering
    df_derivadas = criar_colunas_derivadas(df_limpo)

    # RF13 — Limpeza com regex
    df = limpar_strings_com_regex(df)

    # RF11 — HOF + lambda: marca vendas acima de R$ 10.000
    df_derivadas = processar_coluna(
        df, "receita_total",
        lambda x: "Alto Impacto" if x > 10_000 else "Padrão"
    )

    # RF05–RF07 via AnalisadorComProjecao (RF09 + RF10)
    analisador = AnalisadorComProjecao("vendas.csv", meses_projecao=3)
    (
        analisador
        .carregar()
        .limpar()
        .transformar()
        .analisar()
        .projetar_tendencia()
        .resumo()
    )

    # RF08 — Visualizações
    gerar_visualizacoes(df_derivadas, analisador.metricas)

    # RF12 — Exportação
    exportar_resultados(analisador.metricas, analisador.clientes, analisador.stats)

    print("\n" + "=" * 55)
    print("  Pipeline concluído com sucesso!")
    print("=" * 55)


main()